In [2]:
import mujoco
import numpy as np

# Load model 
model = mujoco.MjModel.from_xml_path("/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/osaka_3p/Osaka_3p_finale.xml")
data = mujoco.MjData(model)

mujoco.mj_forward(model, data)


porte = {
    "sinistra": {
        "joint_name": "giunto_porta_sx",
        "site_name": "target_manigliasx", 
        "angolo_finale": -1.658           
    },
    "destra": {
        "joint_name": "giunto_porta_dx",
        "site_name": "target_manigliadx",   
        "angolo_finale": 1.658           
    }
}

steps = 100


for lato, info in porte.items():
    trajectory = []
    
    # ID
    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, info["site_name"])
    joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, info["joint_name"])
    qpos_adr = model.jnt_qposadr[joint_id]

    # Joint Coordinates
    joint_center = data.xanchor[joint_id].copy()
    Xj, Yj = joint_center[0], joint_center[1]

    pos_handle = data.site_xpos[site_id].copy()
    Xh, Yh = pos_handle[0], pos_handle[1]

    # calculate radius
    radius = np.sqrt((Xh - Xj)**2 + (Yh - Yj)**2)
    # save points
    parameters = np.array([[Xj, Yj, radius]])
    np.savetxt(f"/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/osaka_3p/traiettorie/parametri_cerchio_{lato}.csv", 
               parameters, delimiter=",", header="Centro_X,Centro_Y,Raggio", comments='')
    
    
    target_angles = np.linspace(0, info["angolo_finale"], steps)
    
    
    mujoco.mj_resetData(model, data)

    for angle in target_angles:
        data.qpos[qpos_adr] = angle
        mujoco.mj_forward(model, data) 
        
        pos = data.site_xpos[site_id].copy()
        
        quat = np.zeros(4)
        mujoco.mju_mat2Quat(quat, data.site_xmat[site_id])
        
        trajectory.append(np.concatenate([pos, quat]))

    
    np.save(f'traiettoria_{lato}.npy', np.array(trajectory))
    np.savetxt(f"/Users/cristianvoltan/Desktop/unipd/tirocinio/CAD/osaka_3p/traiettorie/traiettoria_{lato}.csv", trajectory, delimiter=",")
    
    print(f"Traiettoria {lato.upper()} salvata con successo! ({len(trajectory)} punti)")

print("Estrazione completata per entrambe le ante.")

Traiettoria SINISTRA salvata con successo! (100 punti)
Traiettoria DESTRA salvata con successo! (100 punti)
Estrazione completata per entrambe le ante.
